In [ ]:
#read in data
import pandas as pd
import numpy as np

df = pd.read_csv('data_cleaned.csv')

In [ ]:
#filter key word
import re

# List of words to remove (case-insensitive)
frequent_words = ['women', 'woman', 'peacekeepers', 'journalists', 'journalist', 'health']

# Combine into regex pattern for whole words only
pattern = r'\b(?:' + '|'.join(re.escape(word) for word in frequent_words) + r')\b'

# Remove words from clean_text
df['clean_text'] = df['clean_text'].str.replace(pattern, '', case=False, regex=True)

# Remove extra whitespace
df['clean_text'] = df['clean_text'].str.replace(r'\s+', ' ', regex=True).str.strip()


In [19]:
df.head()

,notes,event_id_cnty,target,clean_text
0,"On 25 April 2025, in Capilla del Monte (Cordob...",ARG16601,gender,capilla del monte cordoba large group people i...
1,"Around 25 April 2025 (as reported), in Salvado...",BRA96908,gender,around reported salvador bahia cv members shot...
2,"On 25 April 2025, about 200 Israelis from the ...",ISR45719,gender,israelis shift movement protested jerusalem ju...
3,"On 25 April 2025, in Leon de los Aldama, Guana...",MEX103000,gender,leon de los aldama guanajuato shot dead armed ...
4,"On 25 April 2025, in Sabanas de Xalostoc, Vera...",MEX103223,gender,sabanas de xalostoc veracruz armed individuals...


Run and set up the logistic regression with td vectorizer

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Fill NaNs with empty strings before vectorizing
df['clean_text'] = df['clean_text'].fillna('')

# 1. Vectorize the text data using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])

# 2. Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['target'])  # y is already integer encoded for sklearn

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Initialize and train logistic regression model
clf = LogisticRegression(multi_class='multinomial', solver='saga', max_iter=1000)
clf.fit(X_train, y_train)

# 5. Predict
y_pred = clf.predict(X_test)

# 6. Evaluation
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)

print(f"Test Accuracy: {accuracy:.4f}")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)


/Users/dmnkallen/miniconda3/envs/ml2025/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Test Accuracy: 0.9175
Confusion Matrix:
 [[16037   473   303    11]
 [  632  8068    78     4]
 [  800   125  3180     3]
 [   64     9     8   630]]

Classification Report:
               precision    recall  f1-score   support

      gender       0.91      0.95      0.93     16824
      health       0.93      0.92      0.92      8782
 journalists       0.89      0.77      0.83      4108
 peacekepers       0.97      0.89      0.93       711

    accuracy                           0.92     30425
   macro avg       0.93      0.88      0.90     30425
weighted avg       0.92      0.92      0.92     30425

